## 29/October/2024

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.applications import ResNet50
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO

In [ ]:
# Constants
VIDEO_FOLDER2 = 'fifteen_twenty_five'  # Update this path
VIDEO_FOLDER = "/Users/jeremiah/video-p-reaction-joketype-modular/data/videos"
FRAME_HEIGHT = 224
FRAME_WIDTH = 224
N_FRAMES = 10  # Number of frames to extract from each video
BATCH_SIZE = 32
EPOCHS = 10
LSTM_UNITS = 128

In [ ]:
# Load metadata
metadata_path = '/Users/jeremiah/video-p-reaction-joketype-modular/data/excel.xlsx'  # Update this path
metadata = pd.read_excel(metadata_path)
video_ids = metadata['VideoID'].tolist()
labels = metadata['HowFunny'].tolist()

In [ ]:
# Load ResNet50 for visual feature extraction
base_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')
feature_extractor = Model(inputs=base_model.input, outputs=base_model.output)

# Load YOLOv8 for pose estimation
pose_model = YOLO("yolov8n-pose")  # Adjust model as needed

def extract_frames(video_path):
    """Extract frames from a video file."""
    frames = []
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file: {video_path}")
        return frames

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    for j in range(N_FRAMES):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_count * j / N_FRAMES))
        ret, frame = cap.read()
        if ret:
            frame = cv2.resize(frame, (FRAME_WIDTH, FRAME_HEIGHT))
            frames.append(frame)
        else:
            print(f"Failed to read frame {j} from video: {video_path}")
            break

    cap.release()
    return frames

def extract_visual_features(frames):
    """Extract visual features from frames using ResNet50."""
    features = []
    for frame in frames:
        frame = np.expand_dims(frame, axis=0)
        feature = feature_extractor.predict(frame)
        features.append(feature.flatten())
    return np.array(features)

def extract_movement_features(frames):
    """Extract movement features based on frame differences."""
    movement_features = []
    for i in range(1, len(frames)):
        # Calculate the difference between consecutive frames
        diff = cv2.absdiff(frames[i], frames[i - 1])
        # Convert to grayscale and calculate the mean intensity
        gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
        mean_intensity = np.mean(gray)
        movement_features.append(mean_intensity)
    # Pad the first frame with zeros
    movement_features.insert(0, 0)
    return np.array(movement_features).reshape(-1, 1)  # Reshape for LSTM

# def extract_pose_features(frames):
#     """Extract pose features using YOLOv8."""
#     pose_features = []
#     for frame in frames:
#         results = pose_model(frame)
#         keypoints = []
#         for result in results:
#             if result.keypoints is not None:
#                 keypoints.append(result.keypoints.numpy().flatten())
#         # Ensure we have two rows of keypoints for two people
#         if len(keypoints) == 2:
#             pose_features.append(np.concatenate(keypoints))
#         else:
#             pose_features.append(np.zeros(34))  # Assuming 17 keypoints per person
#     return np.array(pose_features)
import numpy as np
import librosa
from moviepy.editor import VideoFileClip

def extract_speech_features(video_path):
    """Extract speech features using MFCC from audio without saving as .wav."""
    # Load the video file
    video_clip = VideoFileClip(video_path)
    
    # Extract audio from the video
    audio = video_clip.audio
    
    # Check if audio is None
    if audio is None:
        print(f"No audio track found in video: {video_path}")
        return None  # Return None or handle as needed

    # Convert audio to numpy array
    audio_array = audio.to_soundarray(fps=22000)  # Use a sample rate of 22000 Hz
    audio_array = audio_array.mean(axis=1)  # Convert to mono by averaging channels

    # Use librosa to compute MFCCs
    mfccs = librosa.feature.mfcc(y=audio_array, sr=22000, n_mfcc=13)
    
    # Return the mean of the MFCCs across time
    return np.mean(mfccs, axis=1)

# def extract_speech_features(video_path):
#     """Extract speech features using MFCC from audio without saving as .wav."""
#     # Load the video file
#     video_clip = VideoFileClip(video_path)
    
#     # Extract audio from the video
#     audio = video_clip.audio
#     # Convert audio to numpy array
#     audio_array = audio.to_soundarray(fps=22000)  # Use a sample rate of 22000 Hz
#     audio_array = audio_array.mean(axis=1)  # Convert to mono by averaging channels

#     # Use librosa to compute MFCCs
#     mfccs = librosa.feature.mfcc(y=audio_array, sr=22000, n_mfcc=13)
    
#     # Return the mean of the MFCCs across time
#     return np.mean(mfccs, axis=1)# Average over time

def extract_interaction_features(frames):
    """Extract interaction features based on frame analysis."""
    interaction_features = []
    for frame in frames:
        # Example interaction feature: count of detected objects
        results = pose_model(frame)
        interaction_count = sum(1 for result in results if result.keypoints is not None)
        interaction_features.append(np.array([interaction_count, np.random.rand(), 
                                              np.random.rand(),
                                                np.random.rand(),
                                                  np.random.rand()]))  # 5-dimensional interaction features
    return np.array(interaction_features)

def process_video(video_id):
    """Process a single video to extract all features."""
    video_path = os.path.join(VIDEO_FOLDER, f"{video_id}")
    frames = extract_frames(video_path)

    if len(frames) == 0:
        return None, None, None, None, None

    visual_features = extract_visual_features(frames)
    movement_features = extract_movement_features(frames)
    #pose_features = extract_pose_features(frames)
    speech_features = extract_speech_features(video_path)
    interaction_features = extract_interaction_features(frames)

    # Debugging: Print shapes of extracted features
    print(f"Video ID: {video_id}")
    print(f"Visual Features Shape: {visual_features.shape}")
    print(f"Movement Features Shape: {movement_features.shape}")
    #print(f"Pose Features Shape: {pose_features.shape}")
    print(f"Speech Features Shape: {speech_features.shape}")
    print(f"Interaction Features Shape: {interaction_features.shape}")

    return visual_features, movement_features, speech_features, interaction_features

# Prepare feature arrays
X_visual, X_movement, X_speech, X_interaction, y_labels = [], [], [], [], []

for video_id, label in zip(video_ids, labels):
    features = process_video(video_id)
    if features[0] is not None:  # Check if features were extracted
        visual, movement, speech, interaction = features
        X_visual.append(visual)
        X_movement.append(movement)
        #X_pose.append(pose)
        X_speech.append(speech)
        X_interaction.append(interaction)
        y_labels.append(label)

# Convert to numpy arrays
X_visual = np.array(X_visual)
X_movement = np.array(X_movement)
#X_pose = np.array(X_pose)
X_speech = np.array(X_speech)
X_interaction = np.array(X_interaction)
y_labels = np.array(y_labels)

# Debugging: Print shapes of feature arrays
print(f"X_visual Shape: {X_visual.shape}")
print(f"X_movement Shape: {X_movement.shape}")
#print(f"X_pose Shape: {X_pose.shape}")
print(f"X_speech Shape: {X_speech.shape}")
print(f"X_interaction Shape: {X_interaction.shape}")
print(f"y_labels Shape: {y_labels.shape}")

# Define LSTM model for each feature type
def create_lstm_model(input_shape):
    inputs = Input(shape=input_shape)
    x = LSTM(LSTM_UNITS, return_sequences=True)(inputs)
    x = LSTM(LSTM_UNITS // 2)(x)
    x = Dropout(0.3)(x)
    return Model(inputs, x)

# Create models for each feature type
model_visual = create_lstm_model((N_FRAMES, X_visual.shape[2]))
model_movement = create_lstm_model((N_FRAMES, X_movement.shape[2]))
#model_pose = create_lstm_model((N_FRAMES, X_pose.shape[1]))  # Pose features have 34 dimensions
model_speech = create_lstm_model((1, X_speech.shape[1]))  # Speech features are not time-series
model_interaction = create_lstm_model((N_FRAMES, X_interaction.shape[2]))

# Get feature representations
visual_representation = model_visual.predict(X_visual)
movement_representation = model_movement.predict(X_movement)
#pose_representation = model_pose.predict(X_pose)
speech_representation = model_speech.predict(X_speech.reshape(-1, 1, X_speech.shape[1]))  # Reshape for LSTM
interaction_representation = model_interaction.predict(X_interaction)

# Concatenate all feature representations
fused_features = np.concatenate([visual_representation, movement_representation, speech_representation, interaction_representation], axis=1)

# Classification model
def create_classification_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = Dense(128, activation='relu')(inputs)
    x = Dropout(0.5)(x)
    x = Dense(num_classes, activation='softmax')(x)
    return Model(inputs, x)

# Create classification model
num_classes = len(np.unique(y_labels))
classification_model = create_classification_model((fused_features.shape[1],), num_classes)
classification_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train classification model
classification_model.fit(fused_features, y_labels, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.2)

# Evaluate model
y_pred = np.argmax(classification_model.predict(fused_features), axis=1)
print("Classification Report:")
print(classification_report(y_labels, y_pred))

# Plot confusion matrix
plt.figure(figsize=(10, 7))
sns.heatmap(confusion_matrix(y_labels, y_pred), annot=True, fmt='d')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.applications import ResNet50
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO

In [ ]:
# Constants
VIDEO_FOLDER = "/Users/jeremiah/video-p-reaction-joketype-modular/data/videos"
FRAME_HEIGHT = 224
FRAME_WIDTH = 224
N_FRAMES = 10  # Number of frames to extract from each video
BATCH_SIZE = 32
EPOCHS = 50
LSTM_UNITS = 128

# Load metadata
metadata_path = '/Users/jeremiah/video-p-reaction-joketype-modular/data/excel.xlsx'  # Update this path
metadata = pd.read_excel(metadata_path)
video_ids = metadata['VideoID'].tolist()
labels = metadata['HowFunny'].tolist()

In [ ]:
# Load ResNet50 for visual feature extraction
base_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')
feature_extractor = Model(inputs=base_model.input, outputs=base_model.output)

# Load YOLOv8 for object detection
object_detection_model = YOLO("yolov8n")  # Adjust model as needed

def extract_frames(video_path):
    """Extract frames from a video file."""
    frames = []
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file: {video_path}")
        return frames

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    for j in range(N_FRAMES):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_count * j / N_FRAMES))
        ret, frame = cap.read()
        if ret:
            frame = cv2.resize(frame, (FRAME_WIDTH, FRAME_HEIGHT))
            frames.append(frame)
        else:
            print(f"Failed to read frame {j} from video: {video_path}")
            break

    cap.release()
    return frames

def extract_visual_features(frames):
    """Extract visual features from frames using ResNet50."""
    features = []
    for frame in frames:
        frame = np.expand_dims(frame, axis=0)
        feature = feature_extractor.predict(frame)
        features.append(feature.flatten())
    return np.array(features)

def extract_movement_features(frames):
    """Extract movement features based on frame differences."""
    movement_features = []
    for i in range(1, len(frames)):
        # Calculate the difference between consecutive frames
        diff = cv2.absdiff(frames[i], frames[i - 1])
        # Convert to grayscale and calculate the mean intensity
        gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
        mean_intensity = np.mean(gray)
        movement_features.append(mean_intensity)
    # Pad the first frame with zeros
    movement_features.insert(0, 0)
    return np.array(movement_features).reshape(-1, 1)  # Reshape for LSTM

def extract_interaction_features(frames):
    """Extract interaction features based on frame analysis."""
    interaction_features = []
    for frame in frames:
        # Example interaction feature: count of detected objects
        results = object_detection_model(frame)
        interaction_count = len(results)  # Count detected objects
        # Placeholder for additional interaction features
        # You can add more innovative features here, such as:
        # - Average distance between detected objects
        # - Interaction score based on object types
        # - Temporal features based on object movement
        interaction_features.append(np.array([interaction_count]))  # Add more features as needed
    return np.array(interaction_features)

def process_video(video_id):
    """Process a single video to extract all features."""
    video_path = os.path.join(VIDEO_FOLDER, f"{video_id}")
    frames = extract_frames(video_path)

    if len(frames) == 0:
        return None, None, None

    visual_features = extract_visual_features(frames)
    movement_features = extract_movement_features(frames)
    interaction_features = extract_interaction_features(frames)

    # Debugging: Print shapes of extracted features
    print(f"Video ID: {video_id}")
    print(f"Visual Features Shape: {visual_features.shape}")
    print(f"Movement Features Shape: {movement_features.shape}")
    print(f"Interaction Features Shape: {interaction_features.shape}")

    return visual_features, movement_features, interaction_features

# Prepare feature arrays
X_visual, X_movement, X_interaction, y_labels = [], [], [], []

for video_id, label in zip(video_ids, labels):
    features = process_video(video_id)
    if features[0] is not None:  # Check if features were extracted
        visual, movement, interaction = features
        X_visual.append(visual)
        X_movement.append(movement)
        X_interaction.append(interaction)
        y_labels.append(label)

# Convert to numpy arrays
X_visual = np.array(X_visual)
X_movement = np.array(X_movement)
X_interaction = np.array(X_interaction)
y_labels = np.array(y_labels)

# Debugging: Print shapes of feature arrays
print(f"X_visual Shape: {X_visual.shape}")
print(f"X_movement Shape: {X_movement.shape}")
print(f"X_interaction Shape: {X_interaction.shape}")
print(f"y_labels Shape: {y_labels.shape}")


# Define LSTM model for each feature type
def create_lstm_model(input_shape):
    inputs = Input(shape=input_shape)
    x = LSTM(LSTM_UNITS, return_sequences=True)(inputs)
    x = LSTM(LSTM_UNITS // 2)(x)
    x = Dropout(0.3)(x)
    return Model(inputs, x)


# Check if the feature arrays are empty
if X_visual.size == 0 or X_movement.size == 0 or X_interaction.size == 0:
    print("One or more feature arrays are empty. Please check the feature extraction process.")
else:
    # Define LSTM model for each feature type
    n_N_FRAMES = 1425  # Ensure this matches the number of frames you expect
    model_visual = create_lstm_model((N_FRAMES, X_visual.shape[2]))
    model_movement = create_lstm_model((N_FRAMES, X_movement.shape[2]))
    model_interaction = create_lstm_model((N_FRAMES, X_interaction.shape[1]))  # Interaction features

    # Get feature representations
    visual_representation = model_visual.predict(X_visual)
    movement_representation = model_movement.predict(X_movement)
    interaction_representation = model_interaction.predict(X_interaction)

    # Concatenate all feature representations
    fused_features = np.concatenate([visual_representation, movement_representation, interaction_representation], axis=1)

# Classification model
def create_classification_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = Dense(128, activation='relu')(inputs)
    x = Dropout(0.5)(x)
    x = Dense(num_classes, activation='softmax')(x)
    return Model(inputs, x)

# Create classification model
num_classes = len(np.unique(y_labels))
classification_model = create_classification_model((fused_features.shape[1],), num_classes)
classification_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train classification model
classification_model.fit(fused_features, y_labels, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.2)

# Evaluate model
y_pred = np.argmax(classification_model.predict(fused_features), axis=1)
print("Classification Report:")
print(classification_report(y_labels, y_pred))

# Plot confusion matrix
plt.figure(figsize=(10, 7))
sns.heatmap(confusion_matrix(y_labels, y_pred), annot=True, fmt='d')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# Only pose Estimations

In [95]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO

In [96]:
# Constants
VIDEO_FOLDER = "/Users/jeremiah/video-p-reaction-joketype-modular/data/videos"
FRAME_HEIGHT = 224
FRAME_WIDTH = 224
N_FRAMES = 10  # Number of frames to extract from each video
BATCH_SIZE = 32
EPOCHS = 50
LSTM_UNITS = 128

# Load metadata
metadata_path = '/Users/jeremiah/video-p-reaction-joketype-modular/data/excel.xlsx'  # Update this path
metadata = pd.read_excel(metadata_path)
video_ids = metadata['VideoID'].tolist()
labels = metadata['HowFunny'].tolist()

In [97]:

# Load YOLOv8 for pose estimation
pose_model = YOLO("yolov8n-pose")  # Adjust model as needed

def extract_frames(video_path):
    """Extract frames from a video file."""
    frames = []
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file: {video_path}")
        return frames

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    for j in range(N_FRAMES):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_count * j / N_FRAMES))
        ret, frame = cap.read()
        if ret:
            frame = cv2.resize(frame, (FRAME_WIDTH, FRAME_HEIGHT))
            frames.append(frame)
        else:
            print(f"Failed to read frame {j} from video: {video_path}")
            break

    cap.release()
    return frames

def extract_pose_features(frames):
    """Extract pose features using YOLOv8."""
    pose_features = []
    for frame in frames:
        results = pose_model(frame)
        keypoints = []
        for result in results:
            if result.keypoints is not None:
                keypoints.append(result.keypoints.numpy().flatten())
        # Ensure we have two rows of keypoints for two people
        if len(keypoints) == 2:
            pose_features.append(np.concatenate(keypoints))
        else:
            pose_features.append(np.zeros(34))  # Assuming 17 keypoints per person
    return np.array(pose_features)

def process_video(video_id):
    """Process a single video to extract pose features."""
    video_path = os.path.join(VIDEO_FOLDER, f"{video_id}")
    frames = extract_frames(video_path)

    if len(frames) == 0:
        return None

    pose_features = extract_pose_features(frames)

    # Debugging: Print shapes of extracted features
    print(f"Video ID: {video_id}")
    print(f"Pose Features Shape: {pose_features.shape}")

    return pose_features

# Prepare feature arrays
X_pose, y_labels = [], []

for video_id, label in zip(video_ids, labels):
    features = process_video(video_id)
    if features is not None:  # Check if features were extracted
        X_pose.append(features)
        y_labels.append(label)

# Convert to numpy arrays
X_pose = np.array(X_pose)
y_labels = np.array(y_labels)

# Debugging: Print shapes of feature arrays
print(f"X_pose Shape: {X_pose.shape}")
print(f"y_labels Shape: {y_labels.shape}")

# Check if the feature arrays are empty
if X_pose.size == 0:
    print("Pose feature array is empty. Please check the feature extraction process.")
else:
    # Define LSTM model for pose features
    #n_N_FRAMES = 1425  # Ensure this matches the number of frames you expect
    model_pose = create_lstm_model((n_N_FRAMES, X_pose.shape[2]))

    # Get feature representations
    pose_representation = model_pose.predict(X_pose)

    # Classification model
    def create_classification_model(input_shape, num_classes):
        inputs = Input(shape=input_shape)
        x = Dense(128, activation='relu')(inputs)
        x = Dropout(0.5)(x)
        x = Dense(num_classes, activation='softmax')(x)
        return Model(inputs, x)

    # Create classification model
    num_classes = len(np.unique(y_labels))
    classification_model = create_classification_model((pose_representation.shape[1],), num_classes)
    classification_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Train classification model
    classification_model.fit(pose_representation, y_labels, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.2)

    # Evaluate model
    y_pred = np.argmax(classification_model.predict(pose_representation), axis=1)
    print("Classification Report:")
    print(classification_report(y_labels, y_pred))

    # Plot confusion matrix
    plt.figure(figsize=(10, 7))
    sns.heatmap(confusion_matrix(y_labels, y_pred), annot=True, fmt='d')
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()


0: 640x640 2 persons, 77.6ms
Speed: 1.5ms preprocess, 77.6ms inference, 3.5ms postprocess per image at shape (1, 3, 640, 640)


AttributeError: 'Keypoints' object has no attribute 'flatten'. See valid attributes below.

    A class for storing and manipulating detection keypoints.

    This class encapsulates functionality for handling keypoint data, including coordinate manipulation,
    normalization, and confidence values.

    Attributes:
        data (torch.Tensor): The raw tensor containing keypoint data.
        orig_shape (Tuple[int, int]): The original image dimensions (height, width).
        has_visible (bool): Indicates whether visibility information is available for keypoints.
        xy (torch.Tensor): Keypoint coordinates in [x, y] format.
        xyn (torch.Tensor): Normalized keypoint coordinates in [x, y] format, relative to orig_shape.
        conf (torch.Tensor): Confidence values for each keypoint, if available.

    Methods:
        cpu(): Returns a copy of the keypoints tensor on CPU memory.
        numpy(): Returns a copy of the keypoints tensor as a numpy array.
        cuda(): Returns 

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Conv1D, LayerNormalization
from tensorflow.keras.applications import ResNet50
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Constants
VIDEO_FOLDER = "/Users/jeremiah/video-p-reaction-joketype-modular/data/videos"  # Update this path
FRAME_HEIGHT = 224
FRAME_WIDTH = 224
N_FRAMES = 30  # Number of frames to extract from each video
BATCH_SIZE = 32
EPOCHS = 10
LSTM_UNITS = 128

In [ ]:
# Load metadata
metadata_path = '/Users/jeremiah/video-p-reaction-joketype-modular/data/excel.xlsx'  # Update this path
metadata = pd.read_excel(metadata_path)
video_ids = metadata['VideoID'].tolist()
labels = metadata['HowFunny'].tolist()

In [ ]:
# Load ResNet50 for visual feature extraction
base_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')
feature_extractor = Model(inputs=base_model.input, outputs=base_model.output)

def extract_frames(video_path):
    """Extract frames from a video file."""
    frames = []
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file: {video_path}")
        return frames

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    for j in range(N_FRAMES):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_count * j / N_FRAMES))
        ret, frame = cap.read()
        if ret:
            frame = cv2.resize(frame, (FRAME_WIDTH, FRAME_HEIGHT))
            frames.append(frame)
        else:
            print(f"Failed to read frame {j} from video: {video_path}")
            break

    cap.release()
    return frames

def extract_visual_features(frames):
    """Extract visual features from frames using ResNet50."""
    features = []
    for frame in frames:
        frame = np.expand_dims(frame, axis=0)
        feature = feature_extractor.predict(frame)
        features.append(feature.flatten())
    return np.array(features)

def extract_movement_features(frames):
    """Extract movement features (placeholder implementation)."""
    # Implement your movement feature extraction logic here
    # For demonstration, we will return random features
    return np.random.rand(N_FRAMES, 10)  # Example: 10-dimensional movement features

def extract_pose_features(frames):
    """Extract pose features (placeholder implementation)."""
    # Implement your pose estimation logic here
    # For demonstration, we will return random features
    return np.random.rand(N_FRAMES, 15)  # Example: 15-dimensional pose features

def extract_speech_features(video_path):
    """Extract speech features using MFCC from audio."""
    audio_path = video_path.replace('.mp4', '.wav')  # Assuming audio is in .wav format
    y, sr = librosa.load(audio_path, sr=None)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    return np.mean(mfccs, axis=1)  # Average over time

def extract_interaction_features(frames):
    """Extract interaction features (placeholder implementation)."""
    # Implement your interaction feature extraction logic here
    # For demonstration, we will return random features
    return np.random.rand(N_FRAMES, 5)  # Example: 5-dimensional interaction features

def process_video(video_id):
    """Process a single video to extract all features."""
    video_path = os.path.join(VIDEO_FOLDER, f"{video_id}.mp4")
    frames = extract_frames(video_path)

    if len(frames) == 0:
        return None, None, None, None, None

    visual_features = extract_visual_features(frames)
    movement_features = extract_movement_features(frames)
    pose_features = extract_pose_features(frames)
    speech_features = extract_speech_features(video_path)
    interaction_features = extract_interaction_features(frames)

    return visual_features, movement_features, pose_features, speech_features, interaction_features

# Prepare feature arrays
X_visual, X_movement, X_pose, X_speech, X_interaction, y_labels = [], [], [], [], [], []

for video_id, label in zip(video_ids, labels):
    features = process_video(video_id)
    if features[0] is not None:  # Check if features were extracted
        visual, movement, pose, speech, interaction = features
        X_visual.append(visual)
        X_movement.append(movement)
        X_pose.append(pose)
        X_speech.append(speech)
        X_interaction.append(interaction)
        y_labels.append(label)

# Convert to numpy arrays
X_visual = np.array(X_visual)
X_movement = np.array(X_movement)
X_pose = np.array(X_pose)
X_speech = np.array(X_speech)
X_interaction = np.array(X_interaction)
y_labels = np.array(y_labels)

# Define LSTM model for each feature type
def create_lstm_model(input_shape):
    inputs = Input(shape=input_shape)
    x = LSTM(LSTM_UNITS, return_sequences=True)(inputs)
    x = LSTM(LSTM_UNITS // 2)(x)
    x = Dropout(0.3)(x)
    return Model(inputs, x)

# Create models for each feature type
model_visual = create_lstm_model((N_FRAMES, X_visual.shape[2]))
model_movement = create_lstm_model((N_FRAMES, X_movement.shape[2]))
model_pose = create_lstm_model((N_FRAMES, X_pose.shape[2]))
model_speech = create_lstm_model((1, X_speech.shape[1]))  # Speech features are not time-series
model_interaction = create_lstm_model((N_FRAMES, X_interaction.shape[2]))

# Get feature representations
visual_representation = model_visual.predict(X_visual)
movement_representation = model_movement.predict(X_movement)
pose_representation = model_pose.predict(X_pose)
speech_representation = model_speech.predict(X_speech.reshape(-1, 1, X_speech.shape[1]))  # Reshape for LSTM
interaction_representation = model_interaction.predict(X_interaction)

# Concatenate all feature representations
fused_features = np.concatenate([visual_representation, movement_representation, pose_representation, speech_representation, interaction_representation], axis=1)

# Classification model
def create_classification_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = Dense(128, activation='relu')(inputs)
    x = Dropout(0.5)(x)
    x = Dense(num_classes, activation='softmax')(x)
    return Model(inputs, x)

# Create classification model
num_classes = len(np.unique(y_labels))
classification_model = create_classification_model((fused_features.shape[1],), num_classes)
classification_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train classification model
classification_model.fit(fused_features, y_labels, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.2)

# Evaluate model
y_pred = np.argmax(classification_model.predict(fused_features), axis=1)
print("Classification Report:")
print(classification_report(y_labels, y_pred))

# Plot confusion matrix
plt.figure(figsize=(10, 7))
sns.heatmap(confusion_matrix(y_labels, y_pred), annot=True, fmt='d')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
import os
import pandas as pd
import numpy as np
import cv2
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Dropout, concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
import dlib
import shap
from sklearn.cross_decomposition import CCA

In [ ]:
# Constants
VIDEO_FOLDER = 'fifteen_twenty_five'
DATA_FILE = '/Users/jeremiah/video-p-reaction-joketype-modular/data/excel.xlsx'
N_FRAMES = 20  # Number of frames to sample from each video

# Load data
data = pd.read_excel(DATA_FILE)

In [ ]:
# Feature extraction functions
def sample_frames(video_path, n_frames):
    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Adjust if video has fewer frames than requested
    n_frames_to_sample = min(n_frames, frame_count)
    
    for i in range(n_frames_to_sample):
        # Set the position of the next frame to be captured
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_count * i / n_frames_to_sample))
        ret, frame = cap.read()
        if ret:
            # Resize frame to a consistent size
            frame = cv2.resize(frame, (224, 224))
            frames.append(frame)
    
    cap.release()
    
    # If the video has fewer frames than `n_frames`, pad the output with the last valid frame
    if len(frames) < n_frames:
        last_frame = frames[-1] if frames else np.zeros((224, 224, 3), dtype=np.uint8)  # Fallback to blank frame
        frames.extend([last_frame] * (n_frames - len(frames)))
    
    return np.array(frames)


def extract_visual_features(frames):
    base_model = VGG16(weights='imagenet', include_top=False)
    features = base_model.predict(preprocess_input(frames))
    return np.mean(features, axis=0).flatten()[:512]  # Ensure 512 features

def extract_facial_features(frames):
    detector = dlib.get_frontal_face_detector()
    predictor = dlib.shape_predictor("shape_pre/shape_predictor_68_face_landmarks.dat")
    facial_features = []
    for frame in frames:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = detector(gray)
        for face in faces:
            shape = predictor(gray, face)
            facial_features.append(np.array([(p.x, p.y) for p in shape.parts()]).flatten())
    return np.mean(facial_features, axis=0) if facial_features else np.zeros(136)


###################### CLeared ##############################################

from moviepy.editor import VideoFileClip
import librosa
import numpy as np

def extract_audio_from_video(video_path, audio_output_path="temp_audio.wav"):
    try:
        video = VideoFileClip(video_path)
        video.audio.write_audiofile(audio_output_path)
        return audio_output_path
    except Exception as e:
        print(f"Error extracting audio from video: {str(e)}")
        return None

def extract_audio_features(video_path):
    audio_path = extract_audio_from_video(video_path)
    if not audio_path:
        return np.zeros(54)

    try:
        y, sr = librosa.load(audio_path)

        # Ensure valid feature extraction with dimension checks
        def safe_feature_extraction(feature_func, *args, **kwargs):
            try:
                feature = feature_func(*args, **kwargs)
                if feature is None or feature.size == 0:
                    return None
                return feature
            except Exception as e:
                print(f"Error extracting {feature_func.__name__}: {str(e)}")
                return None

        # Extract features safely
        mfcc = safe_feature_extraction(librosa.feature.mfcc, y=y, sr=sr, n_mfcc=13)
        spectral_centroid = safe_feature_extraction(librosa.feature.spectral_centroid, y=y, sr=sr)
        chroma = safe_feature_extraction(librosa.feature.chroma_stft, y=y, sr=sr)

        # Validate dimensions before concatenation
        features = []
        if mfcc is not None:
            features.append(np.mean(mfcc, axis=1))
            features.append(np.std(mfcc, axis=1))
        if spectral_centroid is not None:
            features.append([np.mean(spectral_centroid)])
            features.append([np.std(spectral_centroid)])
        if chroma is not None:
            features.append(np.mean(chroma, axis=1))
            features.append(np.std(chroma, axis=1))

        # Concatenate all features, ensuring proper dimensions
        if features:
            features = np.concatenate(features)
        else:
            features = np.zeros(54)

        return features[:54]  # Ensure 54 features
    except Exception as e:
        print(f"Error extracting audio features: {str(e)}")
        return np.zeros(54)
    
    
def extract_interaction_features(frames):
    detector = dlib.get_frontal_face_detector()
    interaction_features = []
    
    for frame in frames:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = detector(gray)
        
        if len(faces) == 2:
            # Get face 1 and face 2
            face1, face2 = faces[0], faces[1]
            
            # Get facial landmarks for both faces
            landmarks1 = predictor(gray, face1)
            landmarks2 = predictor(gray, face2)
            
            # Extract coordinates for key facial points (nose tips)
            face1_center = (landmarks1.part(30).x, landmarks1.part(30).y)  # Nose tip of face1
            face2_center = (landmarks2.part(30).x, landmarks2.part(30).y)  # Nose tip of face2
            
            # Calculate Euclidean distance between the two faces (nose tips)
            distance = np.sqrt((face1_center[0] - face2_center[0])**2 + 
                               (face1_center[1] - face2_center[1])**2)
            
            # Calculate relative head orientation by comparing nose tips to eye positions
            face1_orientation = landmarks1.part(36).x - landmarks1.part(45).x  # Eye corners distance
            face2_orientation = landmarks2.part(36).x - landmarks2.part(45).x
            
            orientation_diff = abs(face1_orientation - face2_orientation)
            
            # Compute more interaction features such as eye contact distance
            eye_distance = np.sqrt((landmarks1.part(42).x - landmarks2.part(39).x)**2 + 
                                   (landmarks1.part(42).y - landmarks2.part(39).y)**2)
            
            # Combine interaction metrics: distance, orientation, eye contact distance
            features = [distance, orientation_diff, eye_distance]
            interaction_features.append(features)
        else:
            # If not exactly 2 faces detected, append a zero feature vector
            interaction_features.append([0, 0, 0])
    
    # Convert interaction_features list into a numpy array (each row is a frame)
    interaction_features = np.array(interaction_features)
    return interaction_features  # Returns the feature matrix for all frames

def extract_temporal_features2(video_path):
    
    # Load the pre-trained shape predictor for facial landmarks
    predictor = dlib.shape_predictor("/Users/jeremiah/video-p-reaction-joketype-modular/shape_pre/shape_predictor_68_face_landmarks.dat")
    
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"Error: Could not open video file {video_path}")
        return np.zeros(6)  # Return zero features if file can't be opened
    
    # Extract basic temporal features
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    duration = frame_count / fps if fps > 0 else 0
    
    # Initialize variables for motion analysis
    prev_frame = None
    total_motion = 0
    valid_frames = 0
    scene_change_count = 0
    
    # Iterate through the frames
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Motion analysis
        if prev_frame is not None:
            # Calculate frame difference for motion
            frame_diff = cv2.absdiff(prev_frame, gray)
            total_motion += np.mean(frame_diff)
            
            # Scene change detection (if the difference is significantly large)
            if np.mean(frame_diff) > 50:  # You can adjust this threshold
                scene_change_count += 1
            
        prev_frame = gray
        valid_frames += 1
    
    cap.release()
    
    # Compute motion metrics
    avg_motion = total_motion / valid_frames if valid_frames > 0 else 0
    avg_frame_time = 1 / fps if fps > 0 else 0
    
    # Return the extracted temporal features (6 features)
    return np.array([frame_count, fps, duration, avg_motion, scene_change_count, avg_frame_time])



####################################################################

def extract_motion_features(frames):
    prev_gray = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
    motion_features = []
    for frame in frames[1:]:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        magnitude, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        motion_features.append(np.mean(magnitude))
        prev_gray = gray
    return np.array(motion_features)

def extract_temporal_features(video_path):
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    duration = frame_count / fps
    cap.release()
    return np.array([frame_count, fps, duration])[:3]  # Ensure 3 features

# Custom transformer for feature extraction
class VideoFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.scaler = StandardScaler()
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        features = []
        visual_features = []
        audio_features = []
        for video_file in X:
            video_path = os.path.join(VIDEO_FOLDER, video_file)
            if not os.path.exists(video_path):
                print(f"Video file not found: {video_path}")
                continue
            
            try:
                frames = sample_frames(video_path, N_FRAMES)
                
                visual_feat = extract_visual_features(frames)
                facial_feat = extract_facial_features(frames)
                audio_feat = extract_audio_features(video_path)
                motion_feat = extract_motion_features(frames)
                interaction_feat = extract_interaction_features(frames)
                temporal_feat = extract_temporal_features(video_path)
                
                # Ensure all features have consistent shapes
                visual_feat = np.resize(visual_feat, (512,))
                facial_feat = np.resize(facial_feat, (136,))
                audio_feat = np.resize(audio_feat, (54,))
                motion_feat = np.resize(motion_feat, (N_FRAMES-1,))
                interaction_feat = np.array([interaction_feat])
                temporal_feat = np.resize(temporal_feat, (6,))
                
                combined_feat = np.concatenate([
                    visual_feat, facial_feat, audio_feat, 
                    motion_feat, interaction_feat, temporal_feat
                ])
                features.append(combined_feat)
                
                visual_features.append(np.concatenate([visual_feat, facial_feat, motion_feat, interaction_feat]))
                audio_features.append(audio_feat)
                
                print(f"Processed {video_file}: Combined feature shape: {combined_feat.shape}")
            except Exception as e:
                print(f"Error processing {video_file}: {str(e)}")
        
        if not features:
            raise ValueError("No valid features extracted from videos")
        
        features_array = np.array(features)
        print(f"Final features array shape: {features_array.shape}")
        
        # Check for NaN or infinite values
        if np.isnan(features_array).any():
            print("Warning: NaN values detected in features")
            features_array = np.nan_to_num(features_array)
        if np.isinf(features_array).any():
            print("Warning: Infinite values detected in features")
            features_array = np.nan_to_num(features_array)
        
        scaled_features = self.scaler.fit_transform(features_array)
        
        return scaled_features, np.array(visual_features), np.array(audio_features)

# Prepare data
X = data['VideoID']
y_joke = data['JokeType']
y_reaction = data['HowFunny']

# Handle missing values
imputer = SimpleImputer(strategy='most_frequent')
y_joke = imputer.fit_transform(y_joke.values.reshape(-1, 1)).ravel()
y_reaction = imputer.fit_transform(y_reaction.values.reshape(-1, 1)).ravel()

# Encode labels
le_joke = LabelEncoder()
le_reaction = LabelEncoder()
y_joke_encoded = le_joke.fit_transform(y_joke)
y_reaction_encoded = le_reaction.fit_transform(y_reaction)

X_train, X_test, y_train_joke, y_test_joke, y_train_reaction, y_test_reaction = train_test_split(
    X, y_joke_encoded, y_reaction_encoded, test_size=0.2, random_state=42
)

# Feature extraction pipeline
feature_pipeline = Pipeline([
    ('feature_extractor', VideoFeatureExtractor()),
])

X_train_features, X_train_visual, X_train_audio = feature_pipeline.fit_transform(X_train)
X_test_features, X_test_visual, X_test_audio = feature_pipeline.transform(X_test)

# Cross-modal analysis using CCA
cca = CCA(n_components=5)
cca.fit(X_train_visual, X_train_audio)
visual_cca, audio_cca = cca.transform(X_train_visual, X_train_audio)

# Analyze correlations
correlations = [np.corrcoef(visual_cca[:, i], audio_cca[:, i])[0, 1] for i in range(5)]

plt.figure(figsize=(10, 6))
plt.bar(range(1, 6), correlations)
plt.title('Cross-modal Correlations')
plt.xlabel('CCA Component')
plt.ylabel('Correlation')
plt.show()

# Build and train the model
input_shape = X_train_features.shape[1]
num_classes_joke = len(le_joke.classes_)
num_classes_reaction = len(le_reaction.classes_)

inputs = Input(shape=(input_shape,))
x = Dense(128, activation='relu')(inputs)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)
output_joke = Dense(num_classes_joke, activation='softmax', name='joke_output')(x)
output_reaction = Dense(num_classes_reaction, activation='softmax', name='reaction_output')(x)

model = Model(inputs=inputs, outputs=[output_joke, output_reaction])
model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train_features, 
    [y_train_joke, y_train_reaction], 
    epochs=50, 
    batch_size=32, 
    validation_split=0.2,
    verbose=1
)

# Evaluate performance
y_pred_joke, y_pred_reaction = model.predict(X_test_features)
y_pred_joke = np.argmax(y_pred_joke, axis=1)
y_pred_reaction = np.argmax(y_pred_reaction, axis=1)

print("Joke Type Classification Report:")
print(classification_report(y_test_joke, y_pred_joke, target_names=le_joke.classes_))
print("\nReaction Type Classification Report:")
print(classification_report(y_test_reaction, y_pred_reaction, target_names=le_reaction.classes_))

# Plot confusion matrices
plt.figure(figsize=(12, 5))
plt.subplot(121)
sns.heatmap(confusion_matrix(y_test_joke, y_pred_joke), annot=True, fmt='d', xticklabels=le_joke.classes_, yticklabels=le_joke.classes_)
plt.title('Confusion Matrix - Joke Type')
plt.subplot(122)
sns.heatmap(confusion_matrix(y_test_reaction, y_pred_reaction), annot=True, fmt='d', xticklabels=le_reaction.classes_, yticklabels=le_reaction.classes_)
plt.title('Confusion Matrix - Reaction Type')
plt.tight_layout()
plt.show()

# Plot training history
plt.figure(figsize=(12, 5))
plt.subplot(121)
plt.plot(history.history['joke_output_accuracy'], label='Train')
plt.plot(history.history['val_joke_output_accuracy'], label='Validation')
plt.title('Joke Type Accuracy')
plt.legend()
plt.subplot(122)
plt.plot(history.history['reaction_output_accuracy'], label='Train')
plt.plot(history.history['val_reaction_output_accuracy'], label='Validation')
plt.title('Reaction Type Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

# Explainability
explainer = shap.DeepExplainer(model, X_train_features[:100])
shap_values = explainer.shap_values([X_test_features[:10], X_test_features[:10]])

plt.figure(figsize=(10, 5))
plt.subplot(121)
shap.summary_plot(shap_values[0][0], X_test_features, plot_type="bar")
plt.title('SHAP Values - Joke Type')
plt.subplot(122)
shap.summary_plot(shap_values[1][0], X_test_features, plot_type="bar")
plt.title('SHAP Values - Reaction Type')
plt.tight_layout()
plt.show()

# Innovative Interpretation: Psychological Feature Importance
feature_types = [
    ('Visual', 512), ('Facial', 136), ('Audio', 54),
    ('Motion', N_FRAMES-1), ('Interaction', 1), ('Temporal', 3)
]

feature_importance_joke = np.mean(np.abs(shap_values[0][0]), axis=0)
feature_importance_reaction = np.mean(np.abs(shap_values[1][0]), axis=0)

psychological_importance = {
    'Joke Type': {},
    'Reaction Type': {}
}

start = 0
for feature_name, feature_length in feature_types:
    end = start + feature_length
    psychological_importance['Joke Type'][feature_name] = np.mean(feature_importance_joke[start:end])
    psychological_importance['Reaction Type'][feature_name] = np.mean(feature_importance_reaction[start:end])
    start = end

plt.figure(figsize=(12, 5))
plt.subplot(121)
plt.bar(psychological_importance['Joke Type'].keys(), psychological_importance['Joke Type'].values())
plt.title('Psychological Feature Importance - Joke Type')
plt.ylabel('Mean |SHAP Value|')
plt.subplot(122)
plt.bar(psychological_importance['Reaction Type'].keys(), psychological_importance['Reaction Type'].values())
plt.title('Psychological Feature Importance - Reaction Type')
plt.ylabel('Mean |SHAP Value|')
plt.tight_layout()
plt.show()

# Cross-modal Analysis Interpretation
print("\nCross-modal Analysis Interpretation:")
for i, corr in enumerate(correlations, 1):
    print(f"Component {i}: Correlation = {corr:.4f}")
    if corr > 0.5:
        print(f"  Strong correlation between audio and visual features in component {i}")
    elif corr > 0.3:
        print(f"  Moderate correlation between audio and visual features in component {i}")
    else:
        print(f"  Weak correlation between audio and visual features in component {i}")

print("\nOverall Cross-modal Insights:")
print("This analysis reveals the extent to which audio and visual features are related")
print("in the context of humor perception. Strong correlations suggest that certain")
print("aspects of joke delivery are consistently expressed across both modalities,")
print("while weak correlations might indicate unique information carried by each modality.")

# Psychological Interpretation
print("\nPsychological Interpretation:")
print("1. Joke Type Prediction:")
for feature, importance in psychological_importance['Joke Type'].items():
    print(f"   - {feature}: {importance:.4f}")
    if feature == 'Visual' and importance > 0.1:
        print("     This suggests that visual cues play a significant role in joke delivery.")
    elif feature == 'Facial' and importance > 0.1:
        print("     Facial expressions seem to be important in conveying the type of joke.")
    elif feature == 'Audio' and importance > 0.1:
        print("     The audio features, including tone and pitch, are crucial in joke classification.")
    elif feature == 'Motion' and importance > 0.1:
        print("     Body language and gestures contribute to the joke type.")
    elif feature == 'Interaction' and importance > 0.1:
        print("     The physical interaction between parent and child influences the joke type.")
    elif feature == 'Temporal' and importance > 0.1:
        print("     The timing and duration of the joke delivery are significant factors.")

print("\n2. Reaction Type Prediction:")
for feature, importance in psychological_importance['Reaction Type'].items():
    print(f"   - {feature}: {importance:.4f}")
    if feature == 'Visual' and importance > 0.1:
        print("     Visual cues are important in determining the child's reaction.")
    elif feature == 'Facial' and importance > 0.1:
        print("     The child's facial expressions are key indicators of their reaction.")
    elif feature == 'Audio' and importance > 0.1:
        print("     Auditory responses (e.g., laughter, verbal reactions) play a significant role.")
    elif feature == 'Motion' and importance > 0.1:
        print("     The child's body language is an important factor in their reaction.")
    elif feature == 'Interaction' and importance > 0.1:
        print("     The physical interaction between parent and child influences the reaction.")
    elif feature == 'Temporal' and importance > 0.1:
        print("     The timing of the reaction relative to the joke is a significant factor.")

print("\nOverall Psychological Insights:")
print("This analysis provides a quantitative measure of the relative importance of different")
print("psychological and sensory factors in humor perception and reaction among children.")
print("It highlights the multimodal nature of humor, showing how visual, auditory, and")
print("interactive elements contribute to both joke delivery and reception.")
print("The cross-modal analysis further reveals how these different modalities interact,")
print("providing a more comprehensive understanding of the humor perception process.")

In [ ]:
def sample_frames2(video_path, n_frames):
    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Adjust if video has fewer frames than requested
    n_frames_to_sample = min(n_frames, frame_count)
    
    for i in range(n_frames_to_sample):
        # Set the position of the next frame to be captured
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_count * i / n_frames_to_sample))
        ret, frame = cap.read()
        if ret:
            # Resize frame to a consistent size
            frame = cv2.resize(frame, (224, 224))
            frames.append(frame)
    
    cap.release()
    
    # If the video has fewer frames than `n_frames`, pad the output with the last valid frame
    if len(frames) < n_frames:
        last_frame = frames[-1] if frames else np.zeros((224, 224, 3), dtype=np.uint8)  # Fallback to blank frame
        frames.extend([last_frame] * (n_frames - len(frames)))
    
    return np.array(frames)


In [ ]:

def extract_visual_features2(frames):
    base_model = VGG16(weights='imagenet', include_top=False)
    features = base_model.predict(preprocess_input(frames))
    return np.mean(features, axis=0).flatten()[:512]  # Ensure 512 features

In [ ]:
vid_1 = "/Users/jeremiah/video-p-reaction-joketype-modular/fifteen_twenty_five/2UWdXP.joke1.rep2.take1.Peekaboo.mp4"

In [ ]:
frame  =  sample_frames2(vid_1, 30)
frame.shape

In [ ]:
print(extract_visual_features2(vid_1))

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:

# Hyperparameters
N_FRAMES = 15  # Number of frames to extract from each video
FRAME_HEIGHT = 224
FRAME_WIDTH = 224
BATCH_SIZE = 32
EPOCHS = 50
LSTM_UNITS = 512

# Constants
VIDEO_FOLDER = 'fifteen_twenty_five'
DATA_FILE = 'data/excel.xlsx'

In [ ]:

data = pd.read_excel(DATA_FILE)
X = data['VideoID']
y_joke = data['JokeType']
y_reaction = data['HowFunny']

# Encode labels
le_joke = LabelEncoder()
le_reaction = LabelEncoder()
y_joke_encoded = le_joke.fit_transform(y_joke)
y_reaction_encoded = le_reaction.fit_transform(y_reaction)

# Split data
X_train, X_test, y_train_joke, y_test_joke, y_train_reaction, y_test_reaction = train_test_split(
    X, y_joke_encoded, y_reaction_encoded, test_size=0.2, random_state=788
)

# Feature extraction
base_model = EfficientNetB0(weights='imagenet', include_top=False, pooling='avg')
feature_extractor = Model(inputs=base_model.input, outputs=base_model.output)

def extract_features(video_files):
    features = []
    for video_file in video_files:
        video_path = os.path.join(VIDEO_FOLDER, video_file)
        if not os.path.exists(video_path):
            print(f"Video file not found: {video_path}")
            continue
        
        cap = cv2.VideoCapture(video_path)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_features = []
        
        if frame_count == 0:
            print(f"No frames found in video: {video_path}")
            continue
        
        for i in range(N_FRAMES):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_count * i / N_FRAMES))
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(frame, (FRAME_WIDTH, FRAME_HEIGHT))
                frame = preprocess_input(frame)
                frame_features.append(feature_extractor.predict(np.expand_dims(frame, axis=0)).flatten())
            else:
                print(f"Failed to read frame {i} from video: {video_path}")
                break
        
        cap.release()
        
        if len(frame_features) == 0:
            print(f"No features extracted from video: {video_path}")
            continue
        
        # Pad or truncate to ensure consistent shape
        if len(frame_features) < N_FRAMES:
            frame_features.extend([np.zeros_like(frame_features[0])] * (N_FRAMES - len(frame_features)))
        features.append(np.array(frame_features[:N_FRAMES]))
    
    if len(features) == 0:
        raise ValueError("No valid features extracted from any videos")
    
    return np.array(features)

# Extract features
X_train_features = extract_features(X_train)
X_test_features = extract_features(X_test)

# Build model
input_shape = X_train_features.shape[1:]
num_classes_joke = len(le_joke.classes_)
num_classes_reaction = len(le_reaction.classes_)

inputs = Input(shape=input_shape)
x = LSTM(LSTM_UNITS, return_sequences=True)(inputs)
x = LSTM(LSTM_UNITS // 2)(x)
x = Dropout(0.1)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.1)(x)
output_joke = Dense(num_classes_joke, activation='softmax', name='joke_output')(x)
output_reaction = Dense(num_classes_reaction, activation='softmax', name='reaction_output')(x)

model = Model(inputs=inputs, outputs=[output_joke, output_reaction])
model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy','accuracy'])


In [ ]:
# Train model
history = model.fit(
    X_train_features, 
    [y_train_joke, y_train_reaction], 
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE, 
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Evaluate model
y_pred_joke, y_pred_reaction = model.predict(X_test_features)
y_pred_joke = np.argmax(y_pred_joke, axis=1)
y_pred_reaction = np.argmax(y_pred_reaction, axis=1)

In [ ]:
# Print classification reports
print("Joke Type Classification Report:")
print(classification_report(y_test_joke, y_pred_joke, target_names=le_joke.classes_))
print("\nReaction Type Classification Report:")
print(classification_report(y_test_reaction, y_pred_reaction, target_names=le_reaction.classes_))

# Plot confusion matrices
plt.figure(figsize=(12, 5))
plt.subplot(121)
sns.heatmap(confusion_matrix(y_test_joke, y_pred_joke), annot=True, fmt='d', xticklabels=le_joke.classes_, yticklabels=le_joke.classes_)
plt.title('Confusion Matrix - Joke Type')
plt.subplot(122)
sns.heatmap(confusion_matrix(y_test_reaction, y_pred_reaction), annot=True, fmt='d', xticklabels=le_reaction.classes_, yticklabels=le_reaction.classes_)
plt.title('Confusion Matrix - Reaction Type')
plt.tight_layout()
plt.show()


In [ ]:

# Plot training history
plt.figure(figsize=(12, 5))
plt.subplot(121)
plt.plot(history.history['joke_output_accuracy'], label='Train')
plt.plot(history.history['val_joke_output_accuracy'], label='Validation')
plt.title('Joke Type Accuracy')
plt.legend()
plt.subplot(122)
plt.plot(history.history['reaction_output_accuracy'], label='Train')
plt.plot(history.history['val_reaction_output_accuracy'], label='Validation')
plt.title('Reaction Type Accuracy')
plt.legend()
plt.tight_layout()
plt.show()